### The Baselines

* **Random Classifier:**
* **Heuristic:**
* **Multi-Class Linear Classifier:** 

### Neural Network

* **Basic Neural Network / Multilayer Perceptron:**
* **Regularized NN:**

### Natural Language Processing:

* **Bag of Words Model:**
* **RNN Sentiment Analysis:**

### Extra

* **Attention Models:** Use attention mechanisms on the titles/descriptions
* **Some Fancy Pretrained Model maybe BERT too**
* **Just run it into an LLM and see how it does**


## Load and Process Data

In [ ]:
# access parquet :D files with pandas
import numpy as np
import pandas as pd
import polars as pl
import torch
import torch.nn as nn
from collections import Counter
from sklearn.compose import ColumnTransformer
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from torch.utils.data import DataLoader, TensorDataset
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer, AutoModelForSequenceClassification, get_linear_schedule_with_warmup
from torch.amp import autocast, GradScaler


df = pl.read_parquet('data/youtube_data.parquet')
df

title,channel_name,daily_rank,daily_movement,weekly_movement,snapshot_date,country,view_count,like_count,comment_count,description,video_id,channel_id,video_tags,kind,publish_date,langauge,publish_day,like_tier
str,cat,u8,i16,i16,"datetime[μs, UTC]",cat,i64,i32,i32,str,str,str,str,cat,"datetime[μs, UTC]",cat,cat,cat
"""Nyasha David - Tsvodi (Officia…","""Nyasha David""",1,0,15,2026-02-18 00:00:00 UTC,"""ZW""",430573,13860,1139,"""#Tsvodi #nyashadavid #mafindif…","""07pinMbPq2Q""","""UCx1LPxEtJWIpvtXTYadAdKg""","""""","""youtube#video""",2026-02-09 00:00:00 UTC,"""und""","""Monday""","""pretty_viral"""
"""WAR MACHINE | Official Trailer…","""Netflix""",2,0,0,2026-02-18 00:00:00 UTC,"""ZW""",12531660,191591,15336,"""During the final stage of U.S.…","""AFuE1LRxm80""","""UCWOA1ZGywLbqmigxE4Qlvuw""","""Alan Ritchson, Alan Ritchson W…","""youtube#video""",2026-02-04 00:00:00 UTC,"""en-US""","""Wednesday""","""really_viral"""
"""10 Seconds vs 1 Hour MODERN MO…","""Cash""",3,0,47,2026-02-18 00:00:00 UTC,"""ZW""",491455,6139,1656,"""Instagram: https://www.instagr…","""EbBjlznDwzk""","""UC0eLBYhxW9HC0P9PXQ73mpQ""","""Cash, Nico, Nico and Cash, Cas…","""youtube#video""",2026-02-16 00:00:00 UTC,"""en-US""","""Monday""","""a_little_viral"""
"""Master H - Impossible (Officia…","""Master H""",4,0,3,2026-02-18 00:00:00 UTC,"""ZW""",532388,14552,1281,"""Artist : Master H Song : Impos…","""TIG5IzusQ24""","""UC5jrU8WxzE16gPTnlqQAqew""","""""","""youtube#video""",2026-02-06 00:00:00 UTC,"""en""","""Friday""","""pretty_viral"""
"""“Spider-Noir” – Authentic Blac…","""Prime Video""",5,0,45,2026-02-18 00:00:00 UTC,"""ZW""",13406903,239798,10614,"""With no power comes no respons…","""HgMbkitzhEM""","""UCQJWtTnAHhEG5w4uN0udnUQ""","""spider noir official teaser, s…","""youtube#video""",2026-02-12 00:00:00 UTC,"""en""","""Thursday""","""really_viral"""
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
"""انهيار الطبيبة هديل الجمل عند …","""شبكة قدس الاخبارية""",46,4,4,2023-10-26 00:00:00 UTC,"""AE""",1064501,8823,815,"""تابعوا #شبكة_قدس عبر المنصات ا…","""OcPejiPT07U""","""UCFFuMGTQXLitNuiU4SIe5vQ""","""فلسطين, القدس, غزة, تاريخ فلسط…","""youtube#video""",2023-10-22 15:04:53 UTC,"""ar""","""Sunday""","""a_little_viral"""
"""بدهم ايانا نترك بيتنا !""","""وليد ونور | Waleed & Noor""",47,3,3,2023-10-26 00:00:00 UTC,"""AE""",413648,36218,2070,"""حساباتنا على مواقع التواصل الإ…","""qX9XD-5ZGIE""","""UCHywC_HXMWAM6-XvjY8gnUw""","""وليد ونور, نور غسان, وليدمقداد…","""youtube#video""",2023-10-22 16:10:45 UTC,"""ar""","""Sunday""","""pretty_viral"""
"""""إسرائيل"" تنزف!.. هل ينهار الا…","""المخبر الاقتصادي - Mokhbir Eqt…",48,2,2,2023-10-26 00:00:00 UTC,"""AE""",2076478,197026,12930,"""لو انت شركة أو تاجر اتواصل مع …","""s7n2wzIWjtY""","""UC4kRorAXuIkyIX6vwXKaLWg""","""المخبر الاقتصادي, اخبار الاقتص…","""youtube#video""",2023-10-19 17:00:08 UTC,"""ar""","""Thursday""","""really_viral"""


In [8]:
# Dedpulicate videos
df = df.drop("like_count")
df = df.sort("snapshot_date")
df = df.unique(subset=["video_id"], keep="last")

# Sort the new unique dataframe by when the videos were published
df = df.sort("publish_date")

# Split by cutoff date
cutoff_date = df['publish_date'].quantile(0.8, "lower")
print(f"Using publish_date {cutoff_date} as the cutoff point.\n")

train_df = df.filter(pl.col("publish_date") <= cutoff_date)
test_df = df.filter(pl.col("publish_date") > cutoff_date)

X_train = train_df.drop("like_tier")
y_train = train_df.get_column("like_tier")

X_test = test_df.drop("like_tier")
y_test = test_df.get_column("like_tier")

X_train_pd = X_train.to_pandas()    
X_test_pd = X_test.to_pandas()

print(f"Training set size: {len(X_train)} unique videos (past)")
print(f"Testing set size:  {len(X_test)} unique videos (future)")

Using publish_date 2025-11-05 00:00:00+00:00 as the cutoff point.

Training set size: 366469 unique videos (past)
Testing set size:  90674 unique videos (future)


In [9]:
if torch.cuda.is_available():
    device = torch.device("cuda")
elif torch.backends.mps.is_available():
    device = torch.device("mps")
else:
    device = torch.device("cpu")

print(f"Using device: {device}")

Using device: mps


#### Random Classifier

In [ ]:
# a random classifier that takes account in class distributions
# Calculate class probabilities directly from the y_train Series
counts_df = y_train.value_counts()
labels = counts_df.get_column("like_tier").to_list()
probs = (counts_df.get_column("count") / len(y_train)).to_list()

random_preds = np.random.choice(
    labels,
    size=len(y_test),
    p=probs
)

print("Random Classifier Evaluation Report:")
print(classification_report(y_test, random_preds, zero_division=0))

### Huristic Classifiers

#### Majority Class

In [ ]:
# find the most common class in the training labels
majority_class = y_train.mode()[0]

# predict that class for every test sample
heuristic_preds = [majority_class] * len(y_test)


print("Majority Class:", majority_class)
print("Huristic Classifier Evaluation Report (Majority Class):")
print(classification_report(y_test, heuristic_preds, zero_division=0))

In [ ]:
heuristic_feature = "view_count" 
tier_labels = ["a_little_viral", "pretty_viral", "really_viral", "super_viral"]

q1_val = 0.70
q2_val = 0.88
q3_val = 0.97

q1 = X_train.get_column(heuristic_feature).quantile(q1_val)
q2 = X_train.get_column(heuristic_feature).quantile(q2_val)
q3 = X_train.get_column(heuristic_feature).quantile(q3_val)

print(f"Predict '{tier_labels[0]}': <= {q1:.0f}")
print(f"Predict '{tier_labels[1]}': > {q1:.0f} and <= {q2:.0f}")
print(f"Predict '{tier_labels[2]}': > {q2:.0f} and <= {q3:.0f}")
print(f"Predict '{tier_labels[3]}': > {q3:.0f}\n")

heuristic_preds_series = X_test.select(
    pl.when(pl.col(heuristic_feature) <= q1).then(pl.lit(tier_labels[0]))
    .when(pl.col(heuristic_feature) <= q2).then(pl.lit(tier_labels[1]))
    .when(pl.col(heuristic_feature) <= q3).then(pl.lit(tier_labels[2]))
    .otherwise(pl.lit(tier_labels[3]))
    .alias("predictions")
).get_column("predictions")

y_test_list = y_test.to_list()
heuristic_preds = heuristic_preds_series.to_list()

print("Heuristic Classifier Evaluation Report (View Count Quantiles):")
print(classification_report(y_test_list, heuristic_preds, zero_division=0))

### Linear Classifiers

#### Torch model
* 50 epochs
* Numeric Features
* SGD optimizer

In [ ]:
EPOCHS = 10

# numeric only first not like_count lmao
numeric_features = ["daily_rank", "daily_movement", "weekly_movement", "view_count", "comment_count"]

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train.select(numeric_features).to_numpy())
X_test_scaled = scaler.transform(X_test.select(numeric_features).to_numpy())

label_map = {
    "a_little_viral": 0, 
    "pretty_viral": 1, 
    "really_viral": 2, 
    "super_viral": 3
}

y_train_mapped = y_train.to_pandas().map(label_map).to_numpy()
y_test_mapped = y_test.to_pandas().map(label_map).to_numpy()

# Convert everything to PyTorch Tensors
X_train_tensor = torch.tensor(X_train_scaled, dtype=torch.float32).to(device)
y_train_tensor = torch.tensor(y_train_mapped, dtype=torch.long).to(device)

X_test_tensor = torch.tensor(X_test_scaled, dtype=torch.float32).to(device)
y_test_tensor = torch.tensor(y_test_mapped, dtype=torch.long).to(device)

# Create Dataloader
train_dataset = TensorDataset(X_train_tensor, y_train_tensor)
train_loader = DataLoader(train_dataset, batch_size=1024, shuffle=True)

# Simple linear classifier
model = nn.Linear(len(numeric_features), len(label_map)).to(device)
criterion = nn.CrossEntropyLoss()
# We'll use SGD to make the baseline less powerful Adam does too much for us
optimizer = torch.optim.SGD(model.parameters(), lr=0.01)

print("Training Linear Classifier...")
for epoch in range(50):
    model.train()
    for X_batch, y_batch in train_loader:
        optimizer.zero_grad()
        outputs = model(X_batch)
        loss = criterion(outputs, y_batch)
        loss.backward()
        optimizer.step()

model.eval()
with torch.no_grad():
    test_outputs = model(X_test_tensor)
    _, predicted = torch.max(test_outputs, 1)
    
    # Move predictions back to CPU for Scikit-Learn
    predicted_np = predicted.cpu().numpy()

print("\nLinear Classifier Numeric Only Evaluation Report:")
target_names = ["a_little_viral", "pretty_viral", "really_viral", "super_viral"]
print(classification_report(y_test_mapped, predicted_np, target_names=target_names, zero_division=0))

#### Scikit LogisticRegression model
* 100 max_iter
* Numeric Features
* LBFGS solver

In [ ]:
numeric_features = ["daily_rank", "daily_movement", "weekly_movement", "view_count", "comment_count"]

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train.select(numeric_features).to_numpy())
X_test_scaled = scaler.transform(X_test.select(numeric_features).to_numpy())

y_train_np = y_train.to_numpy()
y_test_np = y_test.to_numpy()

print("Training Scikit-Learn Linear Classifier...")

model = LogisticRegression(
    solver='lbfgs',      # Standard solver for multi-class
    max_iter=100,       # Gives it enough time to converge
    random_state=42      # Keeps results reproducible
)
model.fit(X_train_scaled, y_train_np)
predicted_np = model.predict(X_test_scaled)

print("\nScikit-Learn Linear Classifier Evaluation Report:")

target_names = ["a_little_viral", "pretty_viral", "really_viral", "super_viral"]
print(classification_report(y_test_np, predicted_np, labels=target_names, zero_division=0))

### Multilayer Perceptron 
* 50 Epochs
* Numerical Features
* Adam Optimizer


In [ ]:
numeric_features = ["daily_rank", "daily_movement", "weekly_movement", "view_count", "comment_count"]

preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), numeric_features),
    ])

X_train_processed = preprocessor.fit_transform(X_train_pd)
X_test_processed = preprocessor.transform(X_test_pd)

input_dim = X_train_processed.shape[1]
print(f"New Input Dimension: {input_dim} features")

label_map = {
    "a_little_viral": 0, 
    "pretty_viral": 1, 
    "really_viral": 2, 
    "super_viral": 3
}

y_train_mapped = y_train.to_pandas().map(label_map).to_numpy()
y_test_mapped = y_test.to_pandas().map(label_map).to_numpy()

X_train_tensor = torch.tensor(X_train_processed, dtype=torch.float32).to(device)
y_train_tensor = torch.tensor(y_train_mapped, dtype=torch.long).to(device)

X_test_tensor = torch.tensor(X_test_processed, dtype=torch.float32).to(device)
y_test_tensor = torch.tensor(y_test_mapped, dtype=torch.long).to(device)

train_dataset = TensorDataset(X_train_tensor, y_train_tensor)
train_loader = DataLoader(train_dataset, batch_size=1024, shuffle=True)


model = nn.Sequential(
            nn.Linear(input_dim, 128),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(128, 64),
            nn.ReLU(),
            nn.Linear(64, 4)
        ).to(device)
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

print("Training MLP Classifier...")
for epoch in range(50):
    model.train()
    for X_batch, y_batch in train_loader:
        optimizer.zero_grad()
        outputs = model(X_batch)
        loss = criterion(outputs, y_batch)
        loss.backward()
        optimizer.step()

model.eval()
with torch.no_grad():
    test_outputs = model(X_test_tensor)
    _, predicted = torch.max(test_outputs, 1)
    predicted_np = predicted.cpu().numpy()

print("\nMLP Classifier Evaluation Report:")
target_names = ["a_little_viral", "pretty_viral", "really_viral", "super_viral"]
print(classification_report(y_test_mapped, predicted_np, target_names=target_names, zero_division=0))

### Multilayer Perceptron 
* 50 Epochs
* Numerical + Categorical Features
* Adam Optimizer


In [ ]:
numeric_features = ["daily_rank", "daily_movement", "weekly_movement", "view_count", "comment_count"]
categorical_features = ["country", "langauge", "publish_day"]

preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), numeric_features),
        ('cat', OneHotEncoder(handle_unknown='ignore', sparse_output=False), categorical_features)
    ])

X_train_processed = preprocessor.fit_transform(X_train_pd)
X_test_processed = preprocessor.transform(X_test_pd)

input_dim = X_train_processed.shape[1]
print(f"New Input Dimension: {input_dim} features")

label_map = {
    "a_little_viral": 0, 
    "pretty_viral": 1, 
    "really_viral": 2, 
    "super_viral": 3
}

y_train_mapped = y_train.to_pandas().map(label_map).to_numpy()
y_test_mapped = y_test.to_pandas().map(label_map).to_numpy()

X_train_tensor = torch.tensor(X_train_processed, dtype=torch.float32).to(device)
y_train_tensor = torch.tensor(y_train_mapped, dtype=torch.long).to(device)

X_test_tensor = torch.tensor(X_test_processed, dtype=torch.float32).to(device)
y_test_tensor = torch.tensor(y_test_mapped, dtype=torch.long).to(device)

train_dataset = TensorDataset(X_train_tensor, y_train_tensor)
train_loader = DataLoader(train_dataset, batch_size=1024, shuffle=True)

model = nn.Sequential(
            nn.Linear(input_dim, 128),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(128, 64),
            nn.ReLU(),
            nn.Linear(64, 4)
        ).to(device)
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

print("Training MLP Classifier...")
for epoch in range(50):
    model.train()
    for X_batch, y_batch in train_loader:
        optimizer.zero_grad()
        outputs = model(X_batch)
        loss = criterion(outputs, y_batch)
        loss.backward()
        optimizer.step()

model.eval()
with torch.no_grad():
    test_outputs = model(X_test_tensor)
    _, predicted = torch.max(test_outputs, 1)
    predicted_np = predicted.cpu().numpy()

print("\nMLP Classifier Numeric + Categorical Evaluation Report:")
target_names = ["a_little_viral", "pretty_viral", "really_viral", "super_viral"]
print(classification_report(y_test_mapped, predicted_np, target_names=target_names, zero_division=0))

### Natural Language Processing

#### Bag of Words + Logistic Regression


In [ ]:
numeric_features = ["daily_rank", "daily_movement", "weekly_movement", "view_count", "comment_count"]
categorical_features = ["country", "langauge", "publish_day"]

# combine title and description into a single text feature
X_train_pd["text"] = X_train_pd["title"] + " " + X_train_pd["description"]
X_test_pd["text"] = X_test_pd["title"] + " " + X_test_pd["description"]

preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), numeric_features),
        ('cat', OneHotEncoder(handle_unknown='ignore', sparse_output=True), categorical_features),
        # using frequency BoW
        ('bow', CountVectorizer(max_features=2000), "text")
    ]
)

X_train_processed = preprocessor.fit_transform(X_train_pd)
X_test_processed = preprocessor.transform(X_test_pd)

input_dim = X_train_processed.shape[1]
print(f"New Input Dimension with BoW: {input_dim} features")

label_map = {
    "a_little_viral": 0, 
    "pretty_viral": 1, 
    "really_viral": 2, 
    "super_viral": 3
}

y_train_mapped = y_train.to_pandas().map(label_map).to_numpy()
y_test_mapped = y_test.to_pandas().map(label_map).to_numpy()

# count examples per class
class_counts = np.bincount(y_train_mapped)
print("Class counts:", class_counts)

# inverse frequency weights to handle class imbalance
class_weights = {i: max(class_counts)/count for i, count in enumerate(class_counts)}
print("Class weights:", class_weights)

clf = LogisticRegression(
    solver='saga',
    max_iter=3000,
    class_weight=class_weights
)

clf.fit(X_train_processed, y_train_mapped)
y_pred = clf.predict(X_test_processed)

print("\nLogistic Regression with BoW Evaluation Report:")
target_names = ["a_little_viral", "pretty_viral", "really_viral", "super_viral"]
print(classification_report(y_test_mapped, y_pred, target_names=target_names, zero_division=0))

### MLP (Numeric + Categorical + BoW)

In [ ]:
numeric_features = ["daily_rank", "daily_movement", "weekly_movement", "view_count", "comment_count"]
categorical_features = ["country", "langauge", "publish_day"]

X_train_pd["text"] = X_train_pd["title"] + " " + X_train_pd["description"]
X_test_pd["text"] = X_test_pd["title"] + " " + X_test_pd["description"]

preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), numeric_features),
        ('cat', OneHotEncoder(handle_unknown='ignore', sparse_output=True), categorical_features),
        ('bow', CountVectorizer(max_features=2000), "text")
    ])

X_train_processed = preprocessor.fit_transform(X_train_pd)
X_test_processed = preprocessor.transform(X_test_pd)

input_dim = X_train_processed.shape[1]
print(f"New Input Dimension with BoW: {input_dim} features")

label_map = {
    "a_little_viral": 0, 
    "pretty_viral": 1, 
    "really_viral": 2, 
    "super_viral": 3
}

y_train_mapped = y_train.to_pandas().map(label_map).to_numpy()
y_test_mapped = y_test.to_pandas().map(label_map).to_numpy()

X_train_tensor = torch.tensor(X_train_processed.toarray(), dtype=torch.float32).to(device)
y_train_tensor = torch.tensor(y_train_mapped, dtype=torch.long).to(device)

X_test_tensor = torch.tensor(X_test_processed.toarray(), dtype=torch.float32).to(device)
y_test_tensor = torch.tensor(y_test_mapped, dtype=torch.long).to(device)

train_dataset = TensorDataset(X_train_tensor, y_train_tensor)
train_loader = DataLoader(train_dataset, batch_size=1024, shuffle=True)

model = nn.Sequential(
            nn.Linear(input_dim, 512),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(512, 128),
            nn.ReLU(),
            nn.Linear(128, 4)
        ).to(device)

criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)


print("Training MLP with Bag of Words...")
for epoch in range(50):
    model.train()
    total_loss = 0
    for X_batch, y_batch in train_loader:
        optimizer.zero_grad()
        outputs = model(X_batch)
        loss = criterion(outputs, y_batch)
        loss.backward()
        optimizer.step()

model.eval()
with torch.no_grad():
    test_outputs = model(X_test_tensor)
    _, predicted = torch.max(test_outputs, 1)
    predicted_np = predicted.cpu().numpy()

print("\nMLP Classifier (Numeric + Categorical + BoW) Evaluation Report:")
target_names = ["a_little_viral", "pretty_viral", "really_viral", "super_viral"]
print(classification_report(y_test_mapped, predicted_np, target_names=target_names, zero_division=0))

#### TF-IDF + Logistic Regression

In [ ]:
numeric_features = ["daily_rank", "daily_movement", "weekly_movement", "view_count", "comment_count"]
categorical_features = ["country", "langauge", "publish_day"]
X_train_pd["text"] = X_train_pd["title"] + " " + X_train_pd["description"]
X_test_pd["text"] = X_test_pd["title"] + " " + X_test_pd["description"]

preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), numeric_features),
        ('cat', OneHotEncoder(handle_unknown='ignore', sparse_output=True), categorical_features),
        ('tfidf', TfidfVectorizer(max_features=2000, sublinear_tf=True), "text")
    ]
)

X_train_processed = preprocessor.fit_transform(X_train_pd)
X_test_processed = preprocessor.transform(X_test_pd)

label_map = {
    "a_little_viral": 0, 
    "pretty_viral": 1, 
    "really_viral": 2, 
    "super_viral": 3
}

y_train_mapped = y_train.to_pandas().map(label_map).to_numpy()
y_test_mapped = y_test.to_pandas().map(label_map).to_numpy()

input_dim = X_train_processed.shape[1]
print(f"New Input Dimension with TF-IDF: {input_dim} features")

# count examples per class
class_counts = np.bincount(y_train_mapped)
print("Class counts:", class_counts)

# inverse frequency weights to handle class imbalance
class_weights = {i: max(class_counts)/count for i, count in enumerate(class_counts)}
print("Class weights:", class_weights)

clf = LogisticRegression(
    solver='saga',
    max_iter=3000,
    class_weight=class_weights
)

clf.fit(X_train_processed, y_train_mapped)
y_pred = clf.predict(X_test_processed)

print("\nLogistic Regression with TF-IDF Evaluation Report:")
target_names = ["a_little_viral", "pretty_viral", "really_viral", "super_viral"]
print(classification_report(y_test_mapped, y_pred, target_names=target_names, zero_division=0))

### MLP (Numeric + Categorical + TF-IDF)

In [ ]:
numeric_features = ["daily_rank", "daily_movement", "weekly_movement", "view_count", "comment_count"]
categorical_features = ["country", "langauge", "publish_day"]

X_train_pd["text"] = X_train_pd["title"] + " " + X_train_pd["description"]
X_test_pd["text"] = X_test_pd["title"] + " " + X_test_pd["description"]

preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), numeric_features),
        ('cat', OneHotEncoder(handle_unknown='ignore', sparse_output=True), categorical_features),
        ('tfidf', TfidfVectorizer(max_features=2000, sublinear_tf=True), "text")
    ])

X_train_processed = preprocessor.fit_transform(X_train_pd)
X_test_processed = preprocessor.transform(X_test_pd)

input_dim = X_train_processed.shape[1]
print(f"New Input Dimension with TF-IDF: {input_dim} features")

label_map = {
    "a_little_viral": 0, 
    "pretty_viral": 1, 
    "really_viral": 2, 
    "super_viral": 3
}

y_train_mapped = y_train.to_pandas().map(label_map).to_numpy()
y_test_mapped = y_test.to_pandas().map(label_map).to_numpy()

X_train_tensor = torch.tensor(X_train_processed.toarray(), dtype=torch.float32).to(device)
y_train_tensor = torch.tensor(y_train_mapped, dtype=torch.long).to(device)

X_test_tensor = torch.tensor(X_test_processed.toarray(), dtype=torch.float32).to(device)
y_test_tensor = torch.tensor(y_test_mapped, dtype=torch.long).to(device)

train_dataset = TensorDataset(X_train_tensor, y_train_tensor)
train_loader = DataLoader(train_dataset, batch_size=1024, shuffle=True)

model = nn.Sequential(
            nn.Linear(input_dim, 512),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(512, 128),
            nn.ReLU(),
            nn.Linear(128, 4)
        ).to(device)

criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)


print("Training MLP with TFIDF...")
for epoch in range(50):
    model.train()
    for X_batch, y_batch in train_loader:
        optimizer.zero_grad()
        outputs = model(X_batch)
        loss = criterion(outputs, y_batch)
        loss.backward()
        optimizer.step()

model.eval()
with torch.no_grad():
    test_outputs = model(X_test_tensor)
    _, predicted = torch.max(test_outputs, 1)
    predicted_np = predicted.cpu().numpy()

print("\nMLP Classifier (Numeric + Categorical + TF-IDF) Evaluation Report:")
target_names = ["a_little_viral", "pretty_viral", "really_viral", "super_viral"]
print(classification_report(y_test_mapped, predicted_np, target_names=target_names, zero_division=0))

### MLP Vader Sentiment Analysis

In [ ]:

X_train_pd["text"] = X_train_pd["title"] + " " + X_train_pd["description"]
X_test_pd["text"] = X_test_pd["title"] + " " + X_test_pd["description"]

analyzer = SentimentIntensityAnalyzer()

def get_vader_score(text):
    return analyzer.polarity_scores(str(text))["compound"]

X_train_pd["sentiment_score"] = X_train_pd["text"].apply(get_vader_score)
X_test_pd["sentiment_score"] = X_test_pd["text"].apply(get_vader_score)

numeric_features = ["daily_rank", "daily_movement", "weekly_movement", "view_count", "comment_count", "sentiment_score"]
categorical_features = ["country", "langauge", "publish_day"]

preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), numeric_features),
        ('cat', OneHotEncoder(handle_unknown='ignore', sparse_output=True), categorical_features),
        # ('tfidf', TfidfVectorizer(max_features=2000, sublinear_tf=True), "text")
    ])

X_train_processed = preprocessor.fit_transform(X_train_pd)
X_test_processed = preprocessor.transform(X_test_pd)

input_dim = X_train_processed.shape[1]
print(f"New Input Dimension with Sentiment: {input_dim} features")

label_map = {
    "a_little_viral": 0, 
    "pretty_viral": 1, 
    "really_viral": 2, 
    "super_viral": 3
}

y_train_mapped = y_train.to_pandas().map(label_map).to_numpy()
y_test_mapped = y_test.to_pandas().map(label_map).to_numpy()

X_train_tensor = torch.tensor(X_train_processed.toarray(), dtype=torch.float32).to(device)
y_train_tensor = torch.tensor(y_train_mapped, dtype=torch.long).to(device)

X_test_tensor = torch.tensor(X_test_processed.toarray(), dtype=torch.float32).to(device)
y_test_tensor = torch.tensor(y_test_mapped, dtype=torch.long).to(device)

train_dataset = TensorDataset(X_train_tensor, y_train_tensor)
train_loader = DataLoader(train_dataset, batch_size=1024, shuffle=True)

model = nn.Sequential(
            nn.Linear(input_dim, 128),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(128, 64),
            nn.ReLU(),
            nn.Linear(64, 4)
        ).to(device)
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

print("Training MLP with Sentiment...")
for epoch in range(50):
    model.train()
    for X_batch, y_batch in train_loader:
        optimizer.zero_grad()
        outputs = model(X_batch)
        loss = criterion(outputs, y_batch)
        loss.backward()
        optimizer.step()

model.eval()
with torch.no_grad():
    test_outputs = model(X_test_tensor)
    _, predicted = torch.max(test_outputs, 1)
    predicted_np = predicted.cpu().numpy()

print("\nMLP Classifier (Numeric + Categorical + Sentiment) Evaluation Report:")
target_names = ["a_little_viral", "pretty_viral", "really_viral", "super_viral"]
print(classification_report(y_test_mapped, predicted_np, target_names=target_names, zero_division=0))


### Feature Attention MLP 

In [ ]:
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
from sklearn.metrics import classification_report

class FeatureAttentionMLP(nn.Module):
    def __init__(self, input_dim, hidden_dim, num_classes, dropout=0.2):
        super().__init__()
        
        self.attn = nn.Sequential(
            nn.Linear(input_dim, input_dim),
            # Removed the Tanh for sigmoid
        )
        
        self.mlp = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim, hidden_dim // 2),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim // 2, num_classes)
        )

    def forward(self, x):
        attn_scores = self.attn(x)
        attn_weights = torch.sigmoid(attn_scores)
        # Tryin sigmoid instead of softmax cause we got a lotta features
        
        attended_x = x * attn_weights
        
        logits = self.mlp(attended_x)
        return logits, attn_weights

input_dim = X_train_tensor.shape[1]
num_classes = len(label_map)
hidden_dim = 64
epochs = 50
batch_size = 1024
lr = 1e-3
weight_decay = 1e-4

train_dataset = TensorDataset(X_train_tensor, y_train_tensor)
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)

model = FeatureAttentionMLP(
    input_dim=input_dim,
    hidden_dim=hidden_dim,
    num_classes=num_classes,
    dropout=0.2
).to(device)

criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=weight_decay)

print("Training Feature-Attention MLP...")
prev_loss = 1000
for epoch in range(epochs):
    model.train()
    total_loss = 0.0
    
    for X_batch, y_batch in train_loader:
        optimizer.zero_grad()
        logits, _ = model(X_batch)
        loss = criterion(logits, y_batch)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    
    print(f"Epoch {epoch+1}/{50}, Loss: {total_loss/len(train_loader):.4f}")
    if abs(prev_loss - (total_loss/len(train_loader))) < 0.002:
        print("Loss stopped improving. Exiting.")
        break
    prev_loss = total_loss/len(train_loader)
        
model.eval()
with torch.no_grad():
    test_logits, test_attn = model(X_test_tensor)
    predicted = torch.argmax(test_logits, dim=1).cpu().numpy()

target_names = ["a_little_viral", "pretty_viral", "really_viral", "super_viral"]
print("\nFeature-Attention MLP Evaluation Report:")
print(classification_report(y_test_tensor.cpu().numpy(), predicted, target_names=target_names, zero_division=0))

feature_names = preprocessor.get_feature_names_out()
avg_attn = test_attn.mean(dim=0).cpu().numpy()

print("\nTop 15 Features Importance (According to Attention):")
feature_importance = sorted(zip(feature_names, avg_attn), key=lambda x: x[1], reverse=True)

for name, weight in feature_importance[:15]:
    print(f"{name:40s} {weight:.4f}")

Training Feature-Attention MLP...
Epoch 1/50, Loss: 0.8163
Epoch 2/50, Loss: 0.5807
Epoch 3/50, Loss: 0.5457
Epoch 4/50, Loss: 0.5329
Epoch 5/50, Loss: 0.5246
Epoch 6/50, Loss: 0.5182
Epoch 7/50, Loss: 0.5126
Epoch 8/50, Loss: 0.5083
Epoch 9/50, Loss: 0.5051
Epoch 10/50, Loss: 0.5023
Epoch 11/50, Loss: 0.4992
Epoch 12/50, Loss: 0.4977
Loss stopped improving. Exiting.

Feature-Attention MLP Evaluation Report:
                precision    recall  f1-score   support

a_little_viral       0.94      0.95      0.94     73784
  pretty_viral       0.66      0.62      0.64     14234
  really_viral       0.66      0.59      0.63      2486
   super_viral       0.49      0.81      0.61       170

      accuracy                           0.89     90674
     macro avg       0.69      0.74      0.71     90674
  weighted avg       0.88      0.89      0.89     90674


Top 15 Most Important Features (According to Attention):
num__view_count                          0.8328
num__comment_count             

### Pretrained title classifier with DistilBERT

In [ ]:
train_texts = ["" if t is None else str(t) for t in train_df.get_column("title").to_list()]
test_texts = ["" if t is None else str(t) for t in test_df.get_column("title").to_list()]

model_name = "distilbert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(model_name)

print("Tokenizing all text upfront (this takes a while)...")
train_encodings = tokenizer(train_texts, truncation=True, padding=True, max_length=64, return_tensors="pt")
test_encodings = tokenizer(test_texts, truncation=True, padding=True, max_length=64, return_tensors="pt")

class FastTitleDataset(Dataset):
    def __init__(self, encodings, labels):
        self.encodings = encodings
        self.labels = labels

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        item = {key: val[idx] for key, val in self.encodings.items()}
        item['labels'] = torch.tensor(int(self.labels[idx]), dtype=torch.long)
        return item

train_dataset = FastTitleDataset(train_encodings, y_train_mapped)
test_dataset = FastTitleDataset(test_encodings, y_test_mapped)

batch_size = 64
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=batch_size)

model = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=4).to(device)

epochs = 3
optimizer = torch.optim.AdamW(model.parameters(), lr=2e-5)
scheduler = get_linear_schedule_with_warmup(optimizer, num_warmup_steps=0, num_training_steps=len(train_loader) * epochs)
scaler = GradScaler('cuda') # Handles the 16-bit scaling

print("Training DistilBERT...")
for epoch in range(epochs):
    model.train()
    total_loss = 0.0
    
    for batch in train_loader:
        batch = {k: v.to(device) for k, v in batch.items()}
        optimizer.zero_grad()
        
        # Do in cuda on colab or it takes 100 years
        with autocast('cuda'):
            outputs = model(**batch)
            loss = outputs.loss
        
        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()
        scheduler.step()
        
        total_loss += loss.item()
        
    print(f"Epoch {epoch+1}/{epochs}, Loss: {total_loss/len(train_loader):.4f}")

model.eval()
preds, true_labels = [], []

with torch.no_grad():
    for batch in test_loader:
        labels = batch["labels"].numpy()
        batch = {k: v.to(device) for k, v in batch.items()}
        
        with autocast('cuda'):
            outputs = model(**batch)
            logits = outputs.logits
            
        batch_preds = torch.argmax(logits, dim=1).cpu().numpy()
        preds.extend(batch_preds)
        true_labels.extend(labels)

target_names = ["a_little_viral", "pretty_viral", "really_viral", "super_viral"]
print("\nDistilBERT Title Classifier Evaluation Report:")
print(classification_report(true_labels, preds, target_names=target_names, zero_division=0))

/Users/mccayruddick/Documents/Computer_Science/CSC487_Final/DeeplearningFinalUwU/.venv/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 100/100 [00:00<00:00, 12467.46it/s]
DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_transform.weight  | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
pre_classifier.bias     | MISSING    | 
classifier.bias         | MISSING    | 
pre_classifier.weight   | MISSING    | 
classifier.weight       | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok 

Training pretrained title model...


KeyboardInterrupt: 